In [1]:
!pip install opencv-python mediapipe numpy pulsectl

In [ ]:
import cv2
import mediapipe as mp
from math import hypot
import numpy as np
from pulsectl import Pulse, PulseVolumeInfo
 
cap = cv2.VideoCapture(2)
 
mpHands = mp.solutions.hands
hands = mpHands.Hands()
mpDraw = mp.solutions.drawing_utils
 
# Initialize PulseAudio control
pulse = Pulse('volume_control')  
while True:
    success, img = cap.read()
    if not success:
        print("Failed to capture image from camera.")
        break
 
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 
    results = hands.process(imgRGB)
 
    lmList = []
    if results.multi_hand_landmarks:
        for handlandmark in results.multi_hand_landmarks:
            for id, lm in enumerate(handlandmark.landmark):
                h, w, _ = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                lmList.append([id, cx, cy])
            mpDraw.draw_landmarks(img, handlandmark, mpHands.HAND_CONNECTIONS)
 
    if lmList:
        x1, y1 = lmList[4][1], lmList[4][2]  # Thumb tip
        x2, y2 = lmList[8][1], lmList[8][2]  # Index finger tip
 
        cv2.circle(img, (x1, y1), 13, (255, 0, 0), cv2.FILLED)
        cv2.circle(img, (x2, y2), 13, (255, 0, 0), cv2.FILLED)
        cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
 
        length = hypot(x2 - x1, y2 - y1)
        
        # Scale length to a range suitable for volume (0 to 1)
        vol = np.interp(length, [30, 350], [0, 1])
 
        try:
            # Get current sink information
            sink_info = pulse.sink_list()[0] 
 
            # Set volume using PulseAudio
            pulse.volume_set_all_chans(sink_info, vol)
 
            # Debug print statements
            print(f"Length: {int(length)}")
            print(f"Setting volume to: {vol}")
 
            # Draw volume bar and percentage text
            volbar = np.interp(length, [30, 350], [400, 150])
            volper = np.interp(length, [30, 350], [0, 100])
            cv2.rectangle(img, (50, 150), (85, 400), (0, 0, 255), 4)
            cv2.rectangle(img, (50, int(volbar)), (85, 400), (0, 0, 255), cv2.FILLED)
            cv2.putText(img, f"{int(volper)}%", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 98), 3)
 
        except Exception as e:
            print(f"Error setting volume: {e}")
 
    cv2.imshow('Hand Gesture Volume Control', img)
 
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
 
cap.release()
cv2.destroyAllWindows()

2024-09-26 16:03:38.591103: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-26 16:03:38.602716: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-26 16:03:38.665545: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-26 16:03:38.719137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-26 16:03:38.828271: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

Length: 20
Setting volume to: 0.0
Length: 63
Setting volume to: 0.10532010385791234
Length: 55
Setting volume to: 0.08035485346480149
Length: 57
Setting volume to: 0.08687932520773033
Length: 36
Setting volume to: 0.020001717020008105
Length: 22
Setting volume to: 0.0
Length: 22
Setting volume to: 0.0
Length: 24
Setting volume to: 0.0
Length: 25
Setting volume to: 0.0
Length: 24
Setting volume to: 0.0
Length: 24
Setting volume to: 0.0
Length: 24
Setting volume to: 0.0
Length: 25
Setting volume to: 0.0
Length: 38
Setting volume to: 0.027804064617354518
Length: 84
Setting volume to: 0.1687871997451028
Length: 103
Setting volume to: 0.2288977374785077
Length: 124
Setting volume to: 0.2938633906678148
Length: 142
Setting volume to: 0.3512855498440546
Length: 156
Setting volume to: 0.39519018039019865
Length: 169
Setting volume to: 0.43496638486526973
Length: 179
Setting volume to: 0.4660525600602412
Length: 185
Setting volume to: 0.4850587173237805
Length: 191
Setting volume to: 0.50341942